# 🫁 Pneumothorax Detection — Classification & Segmentation

Binary classification (healthy vs. pneumothorax) and pixel-wise segmentation on chest X-rays from the [SIIM-ACR Pneumothorax Segmentation dataset](https://www.kaggle.com/c/siim-acr-pneumothorax-segmentation).

| Model | Task | Notes |
|---|---|---|
| ResNet-18 | Classification | ImageNet pre-training |
| DenseNet-121 | Classification | CheXNet-inspired MLP head |
| Ensemble | Classification | Soft vote (avg. probabilities) |

> **Colab users:** Enable GPU via *Runtime → Change runtime type → T4 GPU* before running.


## 1 · Setup & Imports

In [ ]:
!pip install pydicom segmentation-models-pytorch albumentations torchsummary -q

import os, sys, json, zipfile, warnings, copy, random, glob
from pathlib import Path
from typing import Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import pydicom
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights, DenseNet121_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    confusion_matrix, roc_curve, auc,
)

warnings.filterwarnings('ignore')
print('All libraries imported successfully.')

## 2 · Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    seed: int = 42
    data_dir: str = '/content/pneumothorax_data'
    img_size: int = 224
    mean: tuple = (0.485, 0.456, 0.406)
    std:  tuple = (0.229, 0.224, 0.225)
    batch_size: int = 32
    num_workers: int = 2
    num_epochs: int = 15
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    dropout_rate: float = 0.5
    patience: int = 3
    threshold: float = 0.5
    val_size: float = 0.15
    test_size: float = 0.15

cfg = Config()
os.makedirs(cfg.data_dir, exist_ok=True)

def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True

set_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} ({p.total_memory/1e9:.1f} GB)')

## 3 · Data Upload & Preparation

In [ ]:
from google.colab import files as colab_files

def upload_csv() -> str:
    """Upload CSV file from local disk to Colab."""
    print('\n  Upload trainSet-rle.csv ...')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No CSV uploaded.')
    fname = next(iter(uploaded))
    dest  = os.path.join(cfg.data_dir, fname)
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])
    print(f'  Saved: {dest}  ({len(uploaded[fname])/1024:.1f} KB)')
    return dest

def upload_dicom_zip() -> str:
    """Upload a ZIP archive of DICOM files and extract it."""
    print('\n  Upload DICOM archive (.zip) ...')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No ZIP uploaded.')
    zname = next(iter(uploaded))
    zpath = os.path.join(cfg.data_dir, zname)
    with open(zpath, 'wb') as f:
        f.write(uploaded[zname])
    extract_dir = os.path.join(cfg.data_dir, 'dicom')
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zpath, 'r') as zf:
        zf.extractall(extract_dir)
    n = len(list(Path(extract_dir).rglob('*.dcm')))
    print(f'  {n} DICOM files extracted to {extract_dir}')
    return extract_dir

# Upload your files below
csv_path  = upload_csv()
dicom_dir = upload_dicom_zip()

# Alternative: mount Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# csv_path  = '/content/drive/MyDrive/<YOUR_PATH>/trainSet-rle.csv'
# dicom_dir = '/content/drive/MyDrive/<YOUR_PATH>/dicom-images-train/'

In [ ]:
df = pd.read_csv(csv_path)
print(f'Rows: {len(df):,}  |  Columns: {list(df.columns)}')

df['has_pneumo'] = df['EncodedPixels'].apply(lambda x: 0 if str(x).strip() == '-1' else 1)
unique_df = df.groupby('ImageId')['has_pneumo'].max().reset_index()

n_total = len(unique_df)
n_pos   = int(unique_df['has_pneumo'].sum())
n_neg   = n_total - n_pos
pos_weight_val = n_neg / n_pos

print(f'\nDataset statistics')
print(f'  Total    : {n_total:,}')
print(f'  Positive : {n_pos:,}  ({n_pos/n_total*100:.1f}%)')
print(f'  Negative : {n_neg:,}  ({n_neg/n_total*100:.1f}%)')
print(f'  Imbalance: {pos_weight_val:.2f} : 1')

train_df, tmp_df = train_test_split(
    unique_df, test_size=cfg.val_size + cfg.test_size,
    stratify=unique_df['has_pneumo'], random_state=cfg.seed,
)
val_df, test_df = train_test_split(
    tmp_df,
    test_size=cfg.test_size / (cfg.val_size + cfg.test_size),
    stratify=tmp_df['has_pneumo'], random_state=cfg.seed,
)
print(f'\nSplits  train: {len(train_df):,} | val: {len(val_df):,} | test: {len(test_df):,}')

seg_df = df[df['has_pneumo'] == 1].copy()
if len(seg_df) > 10:
    seg_train_df, seg_val_df = train_test_split(seg_df, test_size=0.2, random_state=cfg.seed)
    print(f'Segmentation  train: {len(seg_train_df):,} | val: {len(seg_val_df):,}')
else:
    seg_train_df = seg_val_df = pd.DataFrame()
    print('Not enough masks for segmentation split.')

## 4 · Dataset & DataLoaders

In [ ]:
def find_dicom(root: str, img_id: str) -> str:
    """Locate a DICOM file by image ID, searching recursively."""
    for ext in ('.dcm', '.DCM'):
        p = os.path.join(root, f'{img_id}{ext}')
        if os.path.exists(p): return p
    for pat in [f'**/{img_id}.dcm', f'**/{img_id}.DCM']:
        hits = glob.glob(os.path.join(root, pat), recursive=True)
        if hits: return hits[0]
    for p in glob.glob(os.path.join(root, '**', '*.dcm'), recursive=True):
        if img_id in os.path.basename(p): return p
    raise FileNotFoundError(f"No DICOM found for '{img_id}' in '{root}'")


def load_dicom_as_pil(root: str, img_id: str, size: int) -> 'Image.Image':
    """Read a DICOM file and return a normalised RGB PIL image."""
    try:
        arr = pydicom.dcmread(find_dicom(root, img_id)).pixel_array.astype(np.float32)
        lo, hi = arr.min(), arr.max()
        arr = (arr - lo) / (hi - lo + 1e-7) * 255.0
        return Image.fromarray(arr.astype(np.uint8)).convert('RGB')
    except Exception as exc:
        print(f'  [WARN] {img_id}: {exc}')
        return Image.new('RGB', (size, size), color=0)


class PneumothoraxClassificationDataset(Dataset):
    """Binary classification dataset — returns (image_tensor, label)."""

    def __init__(self, dataframe, root_dir, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.root_dir  = root_dir
        self.transform = transform
        self.image_ids = self.df['ImageId'].values
        self.labels    = self.df['has_pneumo'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img   = load_dicom_as_pil(self.root_dir, self.image_ids[idx], cfg.img_size)
        label = self.labels[idx]
        if self.transform: img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)


_norm = transforms.Normalize(mean=cfg.mean, std=cfg.std)
train_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), _norm,
])
val_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(), _norm,
])

lkw = dict(num_workers=cfg.num_workers, pin_memory=(device.type == 'cuda'))
train_loader = DataLoader(PneumothoraxClassificationDataset(train_df, dicom_dir, train_tf),
                          batch_size=cfg.batch_size, shuffle=True, drop_last=True, **lkw)
val_loader   = DataLoader(PneumothoraxClassificationDataset(val_df, dicom_dir, val_tf),
                          batch_size=cfg.batch_size, shuffle=False, **lkw)
test_loader  = DataLoader(PneumothoraxClassificationDataset(test_df, dicom_dir, val_tf),
                          batch_size=cfg.batch_size, shuffle=False, **lkw)
print(f'Batches  train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}')

## 5 · Model Definitions

In [ ]:
class ResNet18Classifier(nn.Module):
    """ResNet-18 fine-tuned for binary classification.

    The pre-trained FC layer is replaced by Dropout + Linear(->1 logit).
    Designed to be used with BCEWithLogitsLoss.
    """
    def __init__(self, pretrained: bool = True, dropout: float = 0.5):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        bb = models.resnet18(weights=weights)
        bb.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(bb.fc.in_features, 1))
        self.backbone = bb

    def forward(self, x): return self.backbone(x).squeeze(-1)


class DenseNet121Classifier(nn.Module):
    """DenseNet-121 with a CheXNet-inspired 3-layer classification head.

    Head architecture:
        GlobalAvgPool -> Dropout(0.50) -> Linear(->512) -> ReLU
                      -> Dropout(0.25) -> Linear(->128) -> ReLU
                      -> Dropout(0.12) -> Linear(->1  logit)
    """
    def __init__(self, pretrained: bool = True, dropout: float = 0.5):
        super().__init__()
        weights = DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        bb = models.densenet121(weights=weights)
        in_f = bb.classifier.in_features
        bb.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_f, 512), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout / 2),
            nn.Linear(512, 128), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout / 4),
            nn.Linear(128, 1),
        )
        self.backbone = bb

    def forward(self, x): return self.backbone(x).squeeze(-1)


model_r = ResNet18Classifier(pretrained=True,  dropout=cfg.dropout_rate).to(device)
model_d = DenseNet121Classifier(pretrained=True, dropout=cfg.dropout_rate).to(device)
print(f'ResNet-18     params: {sum(p.numel() for p in model_r.parameters()):,}')
print(f'DenseNet-121  params: {sum(p.numel() for p in model_d.parameters()):,}')

## 6 · Metrics & Training Functions

In [ ]:
def compute_metrics(preds, labels, probs=None) -> dict:
    """Compute a full set of binary-classification metrics.

    Args:
        preds:  Binary predictions array (0 or 1).
        labels: Ground-truth labels array (0 or 1).
        probs:  Predicted probabilities -- required for AUROC computation.

    Returns:
        Dict with keys: accuracy, recall, ppv, f1_score, specificity, npv,
        auroc, confusion_matrix.
    """
    pn, ln = np.asarray(preds), np.asarray(labels)
    rec  = recall_score(ln, pn, zero_division=0)
    ppv  = precision_score(ln, pn, zero_division=0)
    f1   = f1_score(ln, pn, zero_division=0)
    acc  = float(np.mean(pn == ln))
    cm   = confusion_matrix(ln, pn)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    spec = tn / (tn + fp + 1e-8)
    npv  = tn / (tn + fn + 1e-8)
    auroc = None
    if probs is not None:
        fpr, tpr, _ = roc_curve(ln, np.asarray(probs))
        auroc = float(auc(fpr, tpr))
    return dict(accuracy=acc, recall=float(rec), ppv=float(ppv), f1_score=float(f1),
                specificity=float(spec), npv=float(npv), auroc=auroc,
                confusion_matrix=dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp)))


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total, preds_all, labels_all, probs_all = 0.0, [], [], []
    for imgs, lbls in tqdm(loader, desc='  train', leave=False):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, lbls)
        loss.backward(); optimizer.step()
        probs = torch.sigmoid(logits).detach()
        total += loss.item() * imgs.size(0)
        preds_all.append((probs > cfg.threshold).float().cpu())
        labels_all.append(lbls.cpu()); probs_all.append(probs.cpu())
    return (total / len(loader.dataset),
            compute_metrics(torch.cat(preds_all).numpy(),
                            torch.cat(labels_all).numpy(),
                            torch.cat(probs_all).numpy()))


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total, preds_all, labels_all, probs_all = 0.0, [], [], []
    for imgs, lbls in tqdm(loader, desc='  eval ', leave=False):
        imgs, lbls = imgs.to(device), lbls.to(device)
        logits = model(imgs)
        total += criterion(logits, lbls).item() * imgs.size(0)
        probs  = torch.sigmoid(logits)
        preds_all.append((probs > cfg.threshold).float().cpu())
        labels_all.append(lbls.cpu()); probs_all.append(probs.cpu())
    pn = torch.cat(preds_all).numpy()
    ln = torch.cat(labels_all).numpy()
    pr = torch.cat(probs_all).numpy()
    return total / len(loader.dataset), compute_metrics(pn, ln, pr), pr, pn, ln


def train_model(model, name, criterion, optimizer, scheduler):
    """Full training loop with early stopping based on validation F1-score."""
    ckpt = f"best_{name.lower().replace('-','_')}.pth"
    hist = {k: [] for k in ('train_loss','val_loss','train_f1','val_f1',
                             'train_acc','val_acc','train_ppv','val_ppv',
                             'train_recall','val_recall','val_auroc')}
    best_f1, best_weights = -1.0, copy.deepcopy(model.state_dict())
    patience_ctr = 0
    best_out = (None, None, None)

    print(f"\n{'='*55}\n  Training  {name}\n{'='*55}")
    for ep in range(1, cfg.num_epochs + 1):
        print(f'\n  Epoch {ep:02d}/{cfg.num_epochs}')
        tr_loss, tr_m = train_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_m, vl_probs, vl_preds, vl_labels = eval_epoch(model, val_loader, criterion)
        if scheduler: scheduler.step(vl_loss)

        hist['train_loss'].append(tr_loss); hist['val_loss'].append(vl_loss)
        hist['train_f1'].append(tr_m['f1_score']); hist['val_f1'].append(vl_m['f1_score'])
        hist['train_acc'].append(tr_m['accuracy']); hist['val_acc'].append(vl_m['accuracy'])
        hist['train_ppv'].append(tr_m['ppv']); hist['val_ppv'].append(vl_m['ppv'])
        hist['train_recall'].append(tr_m['recall']); hist['val_recall'].append(vl_m['recall'])
        hist['val_auroc'].append(vl_m['auroc'])

        print(f"  Loss {tr_loss:.4f}/{vl_loss:.4f} | F1 {tr_m['f1_score']:.4f}/{vl_m['f1_score']:.4f}"
              f" | PPV {tr_m['ppv']:.4f}/{vl_m['ppv']:.4f} | Recall {tr_m['recall']:.4f}/{vl_m['recall']:.4f}")

        if vl_m['f1_score'] > best_f1:
            best_f1 = vl_m['f1_score']
            best_weights = copy.deepcopy(model.state_dict())
            best_out = (vl_probs, vl_preds, vl_labels)
            patience_ctr = 0
            torch.save({'epoch': ep, 'model_state_dict': model.state_dict(),
                        'val_f1': best_f1, 'history': hist}, ckpt)
            print(f'  Checkpoint saved  (val F1 = {best_f1:.4f})')
        else:
            patience_ctr += 1
            print(f'  No improvement ({patience_ctr}/{cfg.patience})')
            if patience_ctr >= cfg.patience:
                print(f'\n  Early stopping after epoch {ep}.')
                break

    model.load_state_dict(best_weights)
    print(f'\n  Best weights restored  (val F1 = {best_f1:.4f})')
    return hist, *best_out

print('Training utilities defined.')

## 7 · Optimisers & Loss Functions

In [ ]:
pos_w = torch.tensor([pos_weight_val], device=device)

crit_r = nn.BCEWithLogitsLoss(pos_weight=pos_w)
opt_r  = optim.Adam(model_r.parameters(), lr=cfg.learning_rate)
sched_r = optim.lr_scheduler.ReduceLROnPlateau(opt_r, mode='min', patience=2, factor=0.5)

crit_d = nn.BCEWithLogitsLoss(pos_weight=pos_w)
opt_d  = optim.AdamW(model_d.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
sched_d = optim.lr_scheduler.ReduceLROnPlateau(opt_d, mode='min', patience=2, factor=0.5)

print(f'BCEWithLogitsLoss  pos_weight = {pos_weight_val:.2f}')
print('ResNet-18    -> Adam optimizer')
print('DenseNet-121 -> AdamW optimizer + weight decay')

## 8 · Training

In [ ]:
hist_r, *_ = train_model(model_r, 'ResNet-18',   crit_r, opt_r, sched_r)
hist_d, *_ = train_model(model_d, 'DenseNet-121', crit_d, opt_d, sched_d)

## 9 · Test-Set Evaluation

In [ ]:
_, test_m_r, test_probs_r, test_preds_r, test_labels_r = eval_epoch(model_r, test_loader, crit_r)
_, test_m_d, test_probs_d, test_preds_d, test_labels_d = eval_epoch(model_d, test_loader, crit_d)

# Soft-voting ensemble (average of probabilities)
ensemble_probs = (test_probs_r + test_probs_d) / 2
ensemble_preds = (ensemble_probs > cfg.threshold).astype(float)
test_m_ens = compute_metrics(ensemble_preds, test_labels_r, ensemble_probs)

# Comparison table
rows = [
    ('Accuracy',              'accuracy'),
    ('PPV (Precision)',       'ppv'),
    ('Recall (Sensitivity)',  'recall'),
    ('F1-Score',              'f1_score'),
    ('Specificity',           'specificity'),
    ('NPV',                   'npv'),
    ('AUROC',                 'auroc'),
]
cmp = pd.DataFrame({
    'Metric':      [r for r, _ in rows],
    'ResNet-18':   [f"{test_m_r.get(k):.4f}"   if test_m_r.get(k)   is not None else 'N/A' for _, k in rows],
    'DenseNet-121':[f"{test_m_d.get(k):.4f}"   if test_m_d.get(k)   is not None else 'N/A' for _, k in rows],
    'Ensemble':    [f"{test_m_ens.get(k):.4f}" if test_m_ens.get(k) is not None else 'N/A' for _, k in rows],
})
print('\nTest-Set Performance\n')
print(cmp.to_string(index=False))

## 10 · Visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Training Curves', fontsize=15, fontweight='bold')
axes = axes.flatten()

for ax, (tk, vk, title) in zip(axes, [
    ('train_loss', 'val_loss',   'Loss'),
    ('val_acc',    None,         'Val Accuracy'),
    ('val_ppv',    None,         'Val PPV'),
    ('val_recall', None,         'Val Recall'),
    ('val_f1',     None,         'Val F1-Score'),
    ('val_auroc',  None,         'Val AUROC'),
]):
    for hist, name, c in [(hist_r, 'ResNet-18', '#4C72B0'), (hist_d, 'DenseNet-121', '#DD8452')]:
        td = [(i, v) for i, v in enumerate(hist.get(tk, [])) if v is not None]
        if td: ax.plot(*zip(*td), label=name, color=c, lw=2)
        if vk:
            vd = [(i, v) for i, v in enumerate(hist.get(vk, [])) if v is not None]
            if vd: ax.plot(*zip(*vd), label=f'{name} val', color=c, lw=2, ls='--')
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
class_labels = ['Healthy', 'Pneumothorax']

for ax, (preds, labels, name) in zip(axes, [
    (test_preds_r,   test_labels_r, 'ResNet-18'),
    (test_preds_d,   test_labels_d, 'DenseNet-121'),
    (ensemble_preds, test_labels_r, 'Ensemble'),
]):
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_labels, yticklabels=class_labels, annot_kws={'size': 13})
    ax.set_title(f'{name}\nConfusion Matrix', fontweight='bold', pad=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
plt.figure(figsize=(8, 7))
for probs, labels, name, color in [
    (test_probs_r,   test_labels_r, 'ResNet-18',   '#4C72B0'),
    (test_probs_d,   test_labels_d, 'DenseNet-121', '#DD8452'),
    (ensemble_probs, test_labels_r, 'Ensemble',    '#55A868'),
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name}  (AUC = {auc(fpr, tpr):.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
plt.xlim([0, 1]); plt.ylim([0, 1.02])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — Test Set', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight'); plt.show()

## 11 · Segmentation Preview (Ground-Truth Masks)

In [ ]:
def decode_rle(rle_str: str, shape: tuple = (1024, 1024)) -> np.ndarray:
    """Decode a run-length encoded mask string into a binary ndarray.

    Args:
        rle_str: RLE string from the CSV. '-1' means no mask present.
        shape:   (height, width) of the original DICOM image.

    Returns:
        Binary uint8 ndarray of the given shape.
    """
    if not rle_str or str(rle_str).strip() in ('-1', 'nan'):
        return np.zeros(shape, dtype=np.uint8)
    try:
        s       = str(rle_str).split()
        starts  = np.asarray(s[0::2], dtype=int) - 1  # 1-indexed -> 0-indexed
        lengths = np.asarray(s[1::2], dtype=int)
        mask    = np.zeros(shape[0] * shape[1], dtype=np.uint8)
        for st, le in zip(starts, lengths):
            mask[st:st + le] = 1
        return mask.reshape(shape).T  # column-major -> row-major
    except Exception:
        return np.zeros(shape, dtype=np.uint8)


if len(seg_df) >= 3:
    sample = seg_df.sample(3, random_state=cfg.seed)
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    fig.suptitle('Segmentation Masks — Ground Truth', fontsize=14, fontweight='bold')

    for i, (_, row) in enumerate(sample.iterrows()):
        img_pil  = load_dicom_as_pil(dicom_dir, row['ImageId'], cfg.img_size)
        orig_w, orig_h = img_pil.size
        mask     = decode_rle(row['EncodedPixels'], shape=(orig_h, orig_w))
        mask_rs  = np.array(
            Image.fromarray(mask * 255).resize((cfg.img_size, cfg.img_size), Image.NEAREST)
        ) > 127
        img_np = np.array(img_pil)
        axes[i, 0].imshow(img_np, cmap='gray')
        axes[i, 0].set_title('X-Ray'); axes[i, 0].axis('off')
        axes[i, 1].imshow(mask_rs, cmap='Reds')
        axes[i, 1].set_title('Ground-Truth Mask'); axes[i, 1].axis('off')
        axes[i, 2].imshow(img_np)
        axes[i, 2].imshow(mask_rs, alpha=0.4, cmap='Reds')
        axes[i, 2].set_title('Overlay'); axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig('segmentation_preview.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print('Not enough segmentation masks to display.')

## 12 · Save & Download Results

In [ ]:
def _to_json_safe(obj):
    if isinstance(obj, dict): return {k: _to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (float, int)): return obj
    if hasattr(obj, 'item'): return obj.item()
    return obj

with open('results.json', 'w') as f:
    json.dump(_to_json_safe({'ResNet18': test_m_r, 'DenseNet121': test_m_d, 'Ensemble': test_m_ens}), f, indent=2)

pd.DataFrame({
    'true_label':       test_labels_r,
    'resnet18_pred':    test_preds_r,
    'resnet18_prob':    test_probs_r,
    'densenet121_pred': test_preds_d,
    'densenet121_prob': test_probs_d,
    'ensemble_pred':    ensemble_preds,
    'ensemble_prob':    ensemble_probs,
}).to_csv('predictions.csv', index=False)

artifacts = [
    'best_resnet_18.pth', 'best_densenet_121.pth',
    'training_curves.png', 'confusion_matrices.png', 'roc_curves.png',
    'segmentation_preview.png', 'results.json', 'predictions.csv',
]
with zipfile.ZipFile('pneumothorax_results.zip', 'w') as zf:
    for fn in artifacts:
        if os.path.exists(fn): zf.write(fn); print(f'  + {fn}')

print('\nAll results saved.')
colab_files.download('pneumothorax_results.zip')